# Mosquito Suitability: Aggregation by Latitude Band
Generates `mosquito_by_lat_band.csv` from `mosquito_suitability.csv` for use in Tableau.

**Optional upgrades included:**
- A: `city_count` column for sample size visualisation
- B: All three thresholds (Early / Moderate / Strict) in one file
- C: `tropic_line` column for Tableau reference line annotation

In [1]:
import pandas as pd
import numpy as np

In [2]:
#  CONFIG
INPUT_FILE  = 'mosquito_suitability.csv'   # adjust path if needed
OUTPUT_FILE = 'mosquito_suitability_latitude_aggregation.csv'
BAND_WIDTH  = 5                            # degrees per latitude band

THRESHOLDS = {
    'Early (0.2)':    0.2,
    'Moderate (0.3)': 0.3,
    'Strict (0.4)':   0.4,
}


In [3]:
df = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df):,} rows — {df['city'].nunique():,} cities")
df.head(3)

Loaded 17,076 rows — 1,418 cities


,city,country,lat,lon,population,month,temp_mean_c,rh_mean_pct,precip_sum_mm,vpd_kpa,photoperiod_h,vpd_score,temp_score_aegypti,temp_score_albopictus,photo_factor_albopictus_temperate_only,suitability_score_aegypti,suitability_score_albopictus,elevation_m,special_interest
0,Tokyo,Japan,35.687,139.7495,37785000,1,4.85,62.3,70.2,0.3250,9.84,1.0,0.0,0.0,0.0023,0.0,0.0,32.0,0
1,Tokyo,Japan,35.687,139.7495,37785000,2,5.74,61.6,68.4,0.3523,10.70,1.0,0.0,0.0,0.0023,0.0,0.0,32.0,0
2,Tokyo,Japan,35.687,139.7495,37785000,3,9.14,63.4,113.3,0.4238,11.73,1.0,0.0,0.0,0.0023,0.0,0.0,32.0,0


In [4]:
# Step 1: season length per city for each threshold
all_bands = []

for threshold_label, threshold_value in THRESHOLDS.items():

    city_data = df.groupby(['city', 'country', 'lat', 'lon', 'elevation_m']).agg(
        season_aegypti    = ('suitability_score_aegypti',    lambda x: (x >= threshold_value).sum()),
        season_albopictus = ('suitability_score_albopictus', lambda x: (x >= threshold_value).sum()),
    ).reset_index()

    city_data['abs_lat']        = city_data['lat'].abs()
    city_data['lat_band']       = (city_data['abs_lat'] // BAND_WIDTH * BAND_WIDTH).astype(int)
    city_data['lat_band_label'] = city_data['lat_band'].apply(lambda x: f"{x}–{x+BAND_WIDTH}°")
    city_data['lat_band_mid']   = city_data['lat_band'] + BAND_WIDTH / 2

    agg = city_data.groupby(['lat_band', 'lat_band_label', 'lat_band_mid']).agg(
        median_albopictus = ('season_albopictus', 'median'),
        median_aegypti    = ('season_aegypti',    'median'),
        mean_albopictus   = ('season_albopictus', 'mean'),
        mean_aegypti      = ('season_aegypti',    'mean'),
        city_count        = ('city', 'count')          # Upgrade A: sample size
    ).reset_index().sort_values('lat_band')

    agg['threshold'] = threshold_label                 # Upgrade B: threshold column
    all_bands.append(agg)

result = pd.concat(all_bands, ignore_index=True)

for col in ['median_albopictus','median_aegypti','mean_albopictus','mean_aegypti']:
    result[col] = result[col].round(2)

# Upgrade C: tropic annotation flag
result['tropic_line'] = result['lat_band_mid'].apply(
    lambda x: 'Tropics boundary (23.5°)' if abs(x - 23.5) < 3 else None
)

print(f"Total rows: {len(result):,} ({len(THRESHOLDS)} thresholds x {result['lat_band'].nunique()} bands)")
result.head(10)

Total rows: 39 (3 thresholds x 13 bands)


,lat_band,lat_band_label,lat_band_mid,median_albopictus,median_aegypti,mean_albopictus,mean_aegypti,city_count,threshold,tropic_line
0,0,0–5°,2.5,12.0,12.0,11.58,10.96,55,Early (0.2),None
1,5,5–10°,7.5,12.0,12.0,11.87,11.46,114,Early (0.2),None
2,10,10–15°,12.5,12.0,12.0,11.37,11.30,89,Early (0.2),None
3,15,15–20°,17.5,12.0,12.0,10.50,9.32,82,Early (0.2),None
4,20,20–25°,22.5,11.0,10.0,10.15,9.61,153,Early (0.2),Tropics boundary (23.5°)
5,25,25–30°,27.5,5.0,7.0,4.53,7.02,194,Early (0.2),None
6,30,30–35°,32.5,4.0,6.0,3.75,5.64,259,Early (0.2),None
7,35,35–40°,37.5,4.0,5.0,3.77,4.65,235,Early (0.2),None
8,40,40–45°,42.5,4.0,4.0,3.78,3.86,119,Early (0.2),None
9,45,45–50°,47.5,4.0,3.0,3.70,2.86,43,Early (0.2),None


In [5]:
# Quick check: Moderate threshold
moderate = result[result['threshold'] == 'Moderate (0.3)'][['lat_band_label','median_albopictus','median_aegypti','city_count']]
print(moderate.to_string(index=False))

lat_band_label  median_albopictus  median_aegypti  city_count
          0–5°               12.0            12.0          55
         5–10°               12.0            12.0         114
        10–15°               12.0            12.0          89
        15–20°               12.0            11.0          82
        20–25°                9.0             9.0         153
        25–30°                5.0             6.0         194
        30–35°                4.0             5.0         259
        35–40°                4.0             4.0         235
        40–45°                4.0             3.0         119
        45–50°                3.0             3.0          43
        50–55°                3.0             1.0          56
        55–60°                3.0             1.0          23
        60–65°                2.0             0.0           1


In [6]:
# Reshape wide → long: one row per lat_band × threshold × species
long_aeg = result[['lat_band','lat_band_label','lat_band_mid','city_count',
                    'threshold','median_aegypti','mean_aegypti','tropic_line']].copy()
long_aeg = long_aeg.rename(columns={'median_aegypti': 'median_season',
                                       'mean_aegypti':   'mean_season'})
long_aeg['species'] = 'Ae. aegypti'

long_alb = result[['lat_band','lat_band_label','lat_band_mid','city_count',
                    'threshold','median_albopictus','mean_albopictus','tropic_line']].copy()
long_alb = long_alb.rename(columns={'median_albopictus': 'median_season',
                                       'mean_albopictus':   'mean_season'})
long_alb['species'] = 'Ae. albopictus'

long = pd.concat([long_aeg, long_alb], ignore_index=True)
long = long[['lat_band','lat_band_label','lat_band_mid','city_count',
             'species','threshold','median_season','mean_season','tropic_line']]
long = long.sort_values(['threshold','species','lat_band']).reset_index(drop=True)

long.to_csv(OUTPUT_FILE, index=False)
print(f'Saved -> {OUTPUT_FILE}  ({len(long):,} rows)')
print(f'Columns: {list(long.columns)}')
print(long.head(6).to_string(index=False))


Saved -> mosquito_suitability_latitude_aggregation.csv  (78 rows)
Columns: ['lat_band', 'lat_band_label', 'lat_band_mid', 'city_count', 'species', 'threshold', 'median_season', 'mean_season', 'tropic_line']
 lat_band lat_band_label  lat_band_mid  city_count     species   threshold  median_season  mean_season              tropic_line
        0           0–5°           2.5          55 Ae. aegypti Early (0.2)           12.0        10.96                     None
        5          5–10°           7.5         114 Ae. aegypti Early (0.2)           12.0        11.46                     None
       10         10–15°          12.5          89 Ae. aegypti Early (0.2)           12.0        11.30                     None
       15         15–20°          17.5          82 Ae. aegypti Early (0.2)           12.0         9.32                     None
       20         20–25°          22.5         153 Ae. aegypti Early (0.2)           10.0         9.61 Tropics boundary (23.5°)
       25         25–30° 